# Stage 4: Embedding Model & Chunking Strategy Selection (Dev Set Grid)

**Task**: Execute the complete **5x5 Factorial Grid** (5 Embedding Models x 5 Chunking Strategies) across the canonical 1,034-page corpus.
**Evaluation Scope**: 80 grounded Development queries (60 single-page + 20 multi-page across all 10 regions) + 5 Development Negative queries for threshold calibration.
**Output Artifact**: `stage4-dev-selection.zip` containing `candidate-lock.json` for locked final verification.

In [ ]:
# 1. Environment & Device Setup
import os
import sys
import time
import json
import torch
from pathlib import Path

# Ensure local benchmark package is in python path
pkg_path = Path("extras/indexing-benchmarks/code").resolve()
if pkg_path.exists() and str(pkg_path.parent) not in sys.path:
    sys.path.insert(0, str(pkg_path.parent))

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("=" * 60)
print("STAGE 4 DEV SELECTION PREFLIGHT")
print("=" * 60)
print(f"Python Version  : {sys.version.split()[0]}")
print(f"PyTorch Version : {torch.__version__}")
print(f"Selected Device : {device}")
if device == "cuda":
    print(f"GPU Model       : {torch.cuda.get_device_name(0)}")
print("=" * 60)

In [ ]:
# 2. Automatic Input Discovery & Validation
from code.models import discover_candidate_models
from code.corpus import load_canonical_corpus
from code.queries import load_retrieval_queries

print("Checking required Kaggle datasets and model assets...")
pages = load_canonical_corpus()
queries = load_retrieval_queries()
models = discover_candidate_models()

dev_grounded = [q for q in queries if q.split == "dev" and q.type != "out_of_corpus"]
dev_negatives = [q for q in queries if q.split == "dev" and q.type == "out_of_corpus"]

print(f"[OK] Canonical Corpus Pages : {len(pages)} (1016 non-empty text pages)")
print(f"[OK] Dev Grounded Queries   : {len(dev_grounded)} (60 single + 20 multi across 10 regions)")
print(f"[OK] Dev Negative Queries   : {len(dev_negatives)} (out-of-corpus abstention tests)")
print(f"[OK] Discovered Models      : {len(models)}/5 candidate models")
for mid, (mtype, mpath) in models.items():
    print(f"     - {mid:<25} -> {mpath}")

assert len(pages) == 1034, f"Expected 1034 pages, found {len(pages)}"
assert len(dev_grounded) == 80, f"Expected 80 dev grounded queries, found {len(dev_grounded)}"
assert len(dev_negatives) == 5, f"Expected 5 dev negative queries, found {len(dev_negatives)}"
assert len(models) == 5, f"Expected 5 models, found {len(models)}"
print("\nPreflight completed successfully. Ready for 5x5 grid execution!")

In [ ]:
# 3. Run Complete 5x5 Dev Grid and Calibrate Abstention
from code.runner import run_stage4_dev_grid

output_dir = Path("results/stage4-dev-selection").resolve()
t0 = time.perf_counter()

print("Starting 5x5 Dev Grid execution across all 25 cells...")
zip_path = run_stage4_dev_grid(output_dir=output_dir, device=device)
total_time = time.perf_counter() - t0

print(f"\nCompleted 5x5 grid in {total_time:.2f} seconds ({total_time/60:.1f} minutes)!")
print(f"Artifact ZIP created: {zip_path}")

In [ ]:
# 4. Display Dev Leaderboard & Locked Winner Summary
lock_file = output_dir / "candidate-lock.json"
grid_file = output_dir / "dev-grid-results.json"

assert lock_file.exists(), "Missing candidate-lock.json!"
assert grid_file.exists(), "Missing dev-grid-results.json!"

lock_data = json.loads(lock_file.read_text())
grid_data = json.loads(grid_file.read_text())

print("=" * 85)
print("STAGE 4 DEV SELECTION LEADERBOARD (Top 10 Configurations)")
print("=" * 85)
print(f"{Rank:<5} {Model:<24} {Strategy:<22} {R@5:<8} {MRR@10:<8} {Coverage@10:<12} {Latency:<8}")
print("-" * 85)
for i, r in enumerate(grid_data[:10], 1):
    print(f"{i:<5} {r[canonical_model_id]:<24} {r[strategy]:<22} {r[single_page_recall@5]:<8.4f} {r[single_page_mrr@10]:<8.4f} {r[multi_page_coverage@10]:<12.4f} {r[single_query_latency_ms]:<8.1f}ms")

print("=" * 85)
print("LOCKED PRODUCTION WINNER FOR FINAL EVIDENCE NOTEBOOK")
print("=" * 85)
print(f"Winning Model          : {lock_data[winning_model_id]}")
print(f"Winning Chunk Strategy : {lock_data[winning_chunk_strategy]}")
print(f"Embedding Dimension    : {lock_data[dimension]}")
print(f"Dev Recall@5           : {lock_data[dev_recall@5]:.4f} (95% CI: {lock_data[dev_recall@5_ci_95]})")
print(f"Dev MRR@10             : {lock_data[dev_mrr@10]:.4f}")
print(f"Abstention Threshold   : {lock_data[abstention_threshold]:.4f}")
print(f"Dev Abstention Stats   : {lock_data.get(abstention_dev_stats)}")
print("=" * 85)
print(f"\nDownloadable Result ZIP: {zip_path} ({zip_path.stat().st_size / 1024:.1f} KB)")